# ZAIC 2026 LifeMate — v3: ASR + file nộp public đầu tiên

Dùng adapter LoRA đã train ở v2 (`MyDrive/ZAIC/ft_fit_adapter`), **không train lại**.
1. So sánh 2 ASR trên audio của val (WER + điểm end-to-end).
2. Chọn ASR tốt hơn → chạy public_test → `submission_public.jsonl` để **nộp leaderboard**.
3. ASR toàn bộ train (dùng để train lại với transcript có nhiễu ASR ở vòng sau).

GPU nào cũng được (L4 khoảng 1–1.5 giờ). Upload `lifemate_baseline.zip` **bản mới** vào `MyDrive/ZAIC/` → **Run all** → nộp `submission_public.jsonl` lên leaderboard và gửi mình `results_v3.zip` + điểm leaderboard.

In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ASR_MODELS = {"turbo": "openai/whisper-large-v3-turbo",   # 809M
              "pho":   "vinai/PhoWhisper-large"}          # 1.55B (LLM 4.02B + 1.55B = 5.57B < 6B)
BATCH      = 32
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ADAPTER    = DRIVE_DIR + "/ft_fit_adapter"
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_v3"
LLM        = "/content/ft_fit/merged"

In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
assert os.path.exists(f"{WORK}/lifemate/asr.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
assert os.path.exists(ADAPTER + "/adapter_config.json"), "Không thấy adapter v2 trong Drive: " + ADAPTER
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA, "| public audio:", len(glob.glob(DATA + "/public_test/audio/*/*.wav")), "wav")

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-1000:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!python -m lifemate.merge --base {BASE_MODEL} --adapter {ADAPTER} --out {LLM} 2>&1 | grep -vi warn | tail -2
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. So sánh ASR trên audio của val (328 mẫu, khoảng 1000 clip)

In [ ]:
%cd {WORK}
for name, m in ASR_MODELS.items():
    !python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {m} --dtype {DTYPE} --out asr_{name}_val.json 2>&1 | grep -vi warn | tail -4
    shutil.copy(f"asr_{name}_val.json.wer.txt", f"{RES}/wer_{name}.txt")

## 2. Điểm end-to-end: LLM đã fine-tune + transcript gold / từng ASR

In [ ]:
%cd {WORK}
def run_eval(tag, transcripts, sets=("val_unseen", "val_seen")):
    sc = {}
    for S in sets:
        !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts {transcripts} --out runs/{tag}_{S} 2>&1 | tail -1
        !python scripts/score.py --root {DATA} --split train --pred runs/{tag}_{S}/submission.jsonl --by-tag > {RES}/score_{tag}_{S}.txt
        !python -m lifemate.report --root {DATA} --split train --pred runs/{tag}_{S}/submission.jsonl --show 6 > {RES}/report_{tag}_{S}.txt
        sc[S] = float(open(f"{RES}/score_{tag}_{S}.txt").readline().split()[-1])
    print(tag, sc); return sc
scores = {"gold": run_eval("gold", "gold")}
for name in ASR_MODELS:
    scores[name] = run_eval(name, f"asr_{name}_val.json")
json.dump(scores, open(f"{RES}/e2e_scores.json", "w"), indent=1)
BEST = max(ASR_MODELS, key=lambda n: sum(scores[n].values()))
BEST_MODEL = ASR_MODELS[BEST]
print("BEST ASR:", BEST, BEST_MODEL)

## 3. Public test → `submission_public.jsonl` (nộp file này lên leaderboard)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split public_test --model {BEST_MODEL} --dtype {DTYPE} --out asr_public.json 2>&1 | grep -vi warn | tail -2
!python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_public.json --out runs/public 2>&1 | tail -1
shutil.copy("runs/public/submission.jsonl", f"{RES}/submission_public.jsonl")
shutil.copy("runs/public/submission.jsonl", os.path.join(DRIVE_DIR, "submission_public.jsonl"))
shutil.copy("asr_public.json", f"{RES}/asr_public.json"); shutil.copy("runs/public/traces.jsonl", f"{RES}/traces_public.jsonl")
n = sum(1 for _ in open("runs/public/submission.jsonl")); print("lines:", n)
%cd {DATA}/public_test
!python -m zaic_tools --split public_test --root .. --validate {RES}/submission_public.jsonl 2>&1 | tail -5 | tee {RES}/validate_public.txt

## 4. ASR toàn bộ train (cho vòng train lại sau)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split train --model {BEST_MODEL} --dtype {DTYPE} --out asr_train_{BEST}.json 2>&1 | grep -vi warn | tail -3
shutil.copy(f"asr_train_{BEST}.json", f"{RES}/asr_train_{BEST}.json")
shutil.copy(f"asr_train_{BEST}.json", os.path.join(DRIVE_DIR, f"asr_train_{BEST}.json"))

## 5. Đóng gói → `results_v3.zip`

In [ ]:
open(f"{RES}/config.txt", "w").write(f"BASE={BASE_MODEL}\nADAPTER=ft_fit (v2)\nBEST_ASR={BEST}\nGPU={GPU_NAME}\n")
z = shutil.make_archive("/content/results_v3", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_v3.zip"))
print("saved", os.path.join(DRIVE_DIR, "results_v3.zip"), round(os.path.getsize(z)/1e6, 1), "MB")
from google.colab import files
files.download(z); files.download(f"{RES}/submission_public.jsonl")